# An Introduction to SMT Solving

The boolean satisfiability problem (SAT) is a classic problem in computer science that concerns determining whether an assignment of the variables that construct a boolean formula exists such that the formula is satisfiable (or *true*). Notably this problem is NP-complete.

Satisfiability modulo theories (SMT) extends the SAT problem to that of determining the satisfiability over formulae involving more complex variables - such as integers, bitvector and even real numbers. These formulae are constructed 
and solved with respect to these theories. This session does not focus on the background of SMT-solving, but rather on the usage of SMT-solvers. For details on the former see https://www.decision-procedures.org/.

SMT-solvers are tools which solve SMT problems by combining SAT solvers with solvers for each theory they cover. These tools are powerful and highly optimised, though SMT is NP-complete at best and often NP-hard. They are generally used to solve quantifier free formulae. Note that the problem $\exists x \phi(x)$ for some formula $\phi(x)$ can be interpretted in a quantifier free context as $\phi(x)$, which is equisatisfiable.


SMT solvers are generally CLI-based tools that read queries from a standard format - [SMT-LIB2](https://smtlib.cs.uiowa.edu/) (though not all solvers are fully compliant with this). Some solvers also have APIs for different languages, such as C++ or Python. This course relies on Z3-py, the Python API to Z3, which is detailed shortly.

If you are familiar with a particular SMT-solver or would like to use a different solver for any reason to please feel free to do so - though Z3-py is strongly recommended.

This introduction combines some fairly trivial problems to familiarise you with the interface alongside two more complicated ones (solving a Sudoku problem and a CEGIS problem). If you're confident feel free to skip the smaller problems.

In [ ]:
from typing import Callable

import z3
import numpy as np
import torch

## Z3

[Z3](https://link.springer.com/chapter/10.1007/978-3-540-78800-3_24) is a versatile and powerful SMT solver created by Microsoft, and supports many underlying theories (types of formulae), such as Linear Integer Arithmetic, Bitvectors or Non-Linear Real Arithmetic.
Variables in Z3 are declared as their type. For instance, a variable that is an integer would be 
`i = z3.Int('i')`, whereas a real number would be declared as `x = z3.Real('x')`.

Mathmatical formulae can be constructed using standard python builtin operations - e.g., $\phi := x^2 > 0$ `: phi = x ** 2 > 0`

Logical operations are performed using the corresponding functions from the Z3-py library. Examples of these opartions include:
* $\phi \wedge \psi$ :  `z3.And(phi,psi)`
* $\phi \vee \psi$ :  `z3.Or(phi,psi)`
* $\neg \phi$ :  `z3.Not(phi)`

Notably, z3.And and z3.Or can take as many arguments as needed such that:
$\bigwedge_{i=1}^{i=n} \phi_i$ = `z3.And(phi_0, ..., phi_n)`.

To check the satisfiablility of formulae with Z3, you can instantiate a solver using `s = z3.Solver`, and add constraints to this using `s.add(phi)`. 
`s.check()` will return `sat` if the constraint is satisfiable, and `unsat` if it is not. `s.model()` can be used to obtain the satisfying assignment of variables if the result is `sat`.


## Boolean Satisfiability

As previously mentioned, SMT-solvers build on SAT-solvers and extend them to additional theories. This means SMT-solvers are able to solve SAT problems, which is where we begin. Typically, formulae are provided to SAT-solvers in conjunctive normal form (CNF), which means they are expressed as a sequence of clauses of disjunctions joined together with conjunctions. In other words, they can be expressed as $\bigwedge_{i=1}^{i=n} \phi_i$ as above, where $\phi_i$ consists only of disjunctions and literals.

Modern SAT solvers are well optimised and can handle very large problems; here, we will consider a few trivial problems to introduce the Z3 interface.

First, here is an example using Z3 to confirm the following Boolean formula is SAT.

$(x \vee x \vee y) \wedge (¬x \vee ¬y \vee ¬y) \wedge (¬x \vee y \vee y)$

In [ ]:
x = z3.Bool('x')
y = z3.Bool('y')
phi1 = z3.Or(x, x, y)
phi2 = z3.Or(z3.Not(x), z3.Not(y), z3.Not(y))
phi3 = z3.Or(z3.Not(x), y, y)
phi = z3.And(phi1, phi2, phi3)
s = z3.Solver()
s.add(phi)
res = s.check()
print("res = {}".format(res))
m = s.model()
print("assignment: {}".format(m))
s.reset() # Removes all constraints added to solver

**Use Z3 to check if the following boolean formulae are sat or unsat.**

* $(x_2 \vee \neg x_3) \wedge x_1 \wedge x_3 \wedge \neg x_2$
* $(x_1 \vee \neg x_2 \vee x_3) \wedge (x_3 \vee x_5 \vee x_4) \wedge (x_4 \vee \neg x_2 \vee x_1)$

## Finding Real solutions to Polynomials
Using Z3 to solve satisfiability problems over the reals is no different to binary variables, but instead we declare the variables as reals. Internally, the solver must use Quantifer Free Nonlinear Real Arithmetic (don't worry, this is handled by Z3). Consider the simple quadratic equation: 

$x^2 + 6x -7 = 0$

**Use Z3 to find a root of this equation.**

**Which roots will Z3 return?**

**How can we ask Z3 to find the other root? Try this.**


Let $f_1(x,y,z), \ f_2(x,y,z), \ f_3(x,y,z)$ be the polynomials:
\begin{align}
f_1(x,y,z) &= x^2y - 2yz + 1 \\
f_2(x,y,z) &= xy^2 + 2x - z^2 \\
f_3(x,y,z) &= y^2z -y^2 + 5.
\end{align}

**Do the graphs of these polynomials intersect? If so, where?**

## Sudoku Solver: QF_LIA

Solving a sudoku puzzle is a good example of finding a satisfying assignment of variables. We simply want an assignment to all cells in the puzzle that satisfies the puzzle constraints. The constraints for a Sudoku puzzle are as follows:
* Each row and column must contain the numbers 1-9 with no repeats.
* Each of the 9 3x3 sub-grids must contain the numbers 1-9 with no repeats.

<img src="res/Sudoku.png" alt="Drawing" style="width: 200px;"/>

All our constraints are linear, and all variables are integers so we require Quantifier-Free Linear Integer Arithmetic (QF-LIA). Notably, this problem could be solved using a SAT solver by encoding the possible values as dummy binary variables - though it results in many more constraints (and QF-LIA is much simpler to use). **The task here is to solve the sudoku puzzle above using Z3.**
There is additional information below on how to get started, or you can try for yourself!

In [ ]:
N = None
state = [[N, N, N, N, N, N, N, N, N],
         [N, 4, 7, N, N, N, 8, N, N],
         [N, N, N, 6, N, 1, 9, N, 7],
         [N, N, N, N, 4, 6, N, N, 8],
         [6, N, 5, 9, N, 2, 3, N, 4],
         [2, N, N, 3, 1, N, N, N, N],
         [3, N, 2, 1, N, 5, N, N, N],
         [N, N, 1, N, N, N, 2, 9, N],
         [N, N, N, N, N, N, N, N, N]]

  

In [ ]:
# Display puzzle solution

model = s.model()

# This assumes z3 vars are named xij for i,j=0,...,9
for var in model:
    i,j = var.name()[1:]
    state[int(j)][int(i)] = model[var]
    
    
print(np.array(state))

### Sudoku Solver: QF_LIA: Further help

Crucial to this task is asserting that a group of variables should all have different values to each other. If there were only two variables, we'd have 1 condition: $x \neq y$; 3 variables gives $(x \neq y) \wedge (x \neq z) \wedge (y \neq z)$.

In general, how many constraints should there be for $n$ variables?

There are several ways to create a function to create these constraints for us. One such way is using a recursion. The base case would be for two variables as above, and each additional call should assert inequality between the one variable and all others. 

Once we have a function to do this, it is simple to provide constraints to Z3 that enforce the Sudoku constraints. Don't forget to provide the solver with the current state of the puzzle!



## Certifying Specifications using SMT

So far, we have only considered one half of SMT-solving: finding satisfying assignments of variables to find solutions to problems. The other side of these tools is their ability to prove (or certify) specifications, by proving that a given specification holds for all possible values of its variables over some domain of interest. Suppose our variables are $x \in \mathbb{R}^n$, and let $\mathcal{F}(x)$ be the formulae we wish to certify over some domain $\mathcal{D}$. Our specification becomes:

$\forall x \in \mathcal{D}, \ \mathcal{F}(x)$ 

**How can we construct an SMT call that certifies the above specification?**

**How can we interpret the output of the SMT call? In other words - what do `solver.check()` and `solver.model()` provide to us?**

Certify the specifications below and find counter-examples if they are not correct.
* $\mathcal{D}_1 = \{x, y : x^2 + y^2 <= 10 \}, \mathcal{F}_1 = x^2 + y ^2 >= 0$
* $\mathcal{D}_2 = \{x, y : -1 < x, x < 1 , -1 > y, y < 1 \}, \mathcal{F}_2 = x^2 + y^2 <= 1$

## Proving Bit Tricks using SMT-based certification

Theories over fixed-length bitvectors allow SMT-solvers to reason over a data type fundamental to software and hardware verification. Low level computer programming involves bit manipulation techniques ([bit-tricks](https://graphics.stanford.edu/~seander/bithacks.html#IntegerMinOrMax)), which can make certain tasks computationally cheaper. 

**In this section, use Z3 to prove that the following bit tricks (written in `c`) are equivalent to the full operation for an `int` in `c`.**

A fixed length bitvector can be declared in Z3 using `z3.BitVec('name', bit_length)` for a variable, or `z3.BitVecVal(value, bit_length)` for a constant. Standard Python bit operators are compatible with the Z3-py interface, and correspond to _signed_ integers.  The unsigned equivalents can be accessed as Z3 functions - e.g. `z3.ULT` is $<$ and `z3.LShR` is $>>$.

1. **Detects if two (signed) integers have opposite signs.**
    * `f` evalautes to true iff x, y have opposite signs
    ```c
    int x, y
    bool f = ((x ^ y) < 0);
    ```


1. **Determines if an integer is a power of two.**
    ```c
    int x;
    bool f;
    f = x && !(x & (x - 1));
    ```


1. **Branchless computation of abs**
    ```c
    int x;
    unsigned int bit_abs;
    int const mask = x >> (sizeof(int) * 8 - 1);

    bit_abs = (x ^ mask) - mask;

    ```

1. **Compute the minimum of two integers without maxing**
    * This one is a bit trickier. You might need to express the final term as an equivalent expression for Z3 to handle it.
    ```c
    int x, y;
    int bit_min;

    bit_min = y ^ ((x ^ y) & -(x < y)));

    ```



## Counter-example Guided Inductive Synthesis

Finally, we can investigate using SMT solvers for synthesis purposes. 

CEGIS (counter-example guided inductive synthesis) is a common approach to synthesise Lyapunov functions (or other synthesis problems). CEGIS relies on two opposing components, a learner and a verifier. The learner uses inductive procedures over a finite set of points to guess a candidate function, which is then checked formally by the verifier. CEGIS works by splitting the $\exists \forall$ into two separate problems. In doing so, the procedure loses completeness (if the function space is not finite).

<img src="res/cegis.png" alt="Drawing" style="width: 400px;"/>


This section contains a toy CEGIS implementation for a common synthesis task in formal verification. We seek to synthesise a Lyapunov function to prove that a continuous-time dynamical model is stable.

In the following section, it might be useful to use [CVC5](https://cvc5.github.io/) instead of Z3. CVC5 has the same interface as Z3$^1$, except it also provides non-polynomial functions, such as `cvc5.Cosine` and `cvc5.Exponential`. The internal variables to the SMT solvers are not compatible, so you cannot mix their functions.


[1] CVC5's pythonic interface is less optimised than Z3's, so it can be slower at times (though the solver itself is at least as good). It can be useful to explicitly set the Solver logic to NRA, as follows `s.solver.setLogic("NRAT")`, where `s` is a CVC5 solver object. 


In [ ]:
from cvc5 import pythonic as cvc5

## Counter-example Guided Synthesis of Lyapunov Functions

From a verification & control perspective, stability is a crucial property of a system. The most common approach to synthesis of Lyapunov functions is using [Sum-of-Squares optimisation](https://ieeexplore.ieee.org/document/1184414) . However, this relies on solving an SDP, and SDP solvers are known to be susceptible to [numerical errors](http://link.springer.com/10.1007/978-3-662-53413-7_21). Instead, we can turn to SMT-solving to certify stability.

Let $f$ be a dynamical system with some equilibrium point $x^*$ (WLOG this can be assumed to be the origin).  A Lyapunov function is a function $V: \mathbb{R}^n \to \mathbb{R}$, $V \in \mathcal{C}^1$ that satisfies the following properties in some domain $\mathcal{D}$:

* $V(0) = 0$,
* $V(x) > 0 \ \forall x \in \mathcal{D} \setminus \{x^*\}$,
* $\langle \nabla V, f(x) \rangle = \dot{V}(x) < 0 \ \forall x \in \mathcal{D} \setminus \{x^*\}$*.

If such a function exists, then the equilibrium point $x^*$ is asymptotically stable within $\mathcal{D}$. 

Synthesing a Lyapunov function reduces to the following Exists-Forall problem:

$\exists \ V(x) \ \forall x \ \phi(x)$,

where $\phi(x)$ denotes the three Lyapunov conditions described above.

Firstly, we must propose a suitable loss function for the gradient descent scheme.

**How can we ensure the first condition holds?**

For the other two conditions, we require a loss that penalises the sign of the function being incorrect. For the second condition, a function that penalises negative values (more negative is worse), and is neutral to (or possibly rewards) positive values. The opposite is then required for the final condition.

**Propose a suitable loss function for the Learner.**

The implementation is currently missing a function that uses SMT to check if the Learner represents a valid Lyapunov functions, and functionality to return counter-examples to the learner.

Complete the implementation to synthesise a Lyapunov function for the following system dynamics:

\begin{align}
\dot{x} &= -x + xy \\
\dot{y} &= -y
\end{align}

with domain $\mathcal{D} = \{x,y : x^2 + y^2 <=1\}$.
<p align="center">
<img src="res/stable-model.png" alt="Drawing" style="width: 400px;"/>
</p>

The following functions are incomplete and must be finished for the implementation to work:
* `Learner.loss`
*  `Learner.learn`
* `Cegis.synthesise`
* `verify`

Note - it might be easier to store the domain as a representation, rather than a formula in CVC5/Z3, until it is needed.

Finally, you will need to propose a suitable activation function for the Lyapunov Neural network, and construct a corresponding nn.Module class. Since we need the activation function to work in both numerical (PyTorch) and symbolic (SMT-solver) domains, we need to write functions that compute the activation function and its derivative symbolically. When choosing an activation function, it's important to remember the conditions required by the Lyapunov function $V$. Provided is an example CubicActivation function to help get you started (using PyTorch for polynomials is overkill, but it's still a powerful GD tool), but cubic functions are not suitable for this task (why?).

**What can we conclude about the region of attraction of the equilibrium point from the Lyapunov function we synthesise?** (The region of attraction is the set of initial conditions that will converge to the equilibrium point).

In [ ]:

class Activation(torch.nn.Module):
    def forward(x:torch.Tensor) -> torch.Tensor:
        raise NotImplementedError
        
    def forward_symbolic(self,x):
        """Evaluates the activation function in terms of SMT solver variables"""
        raise NotImplementedError

    def gradient_symbolic(self, x):
        """Evalutate the derivative of the actiation function in terms of SMT solver variables"""
        raise NotImplementedError

        
class CubicActivation(torch.nn.Module):
    def forward(self, x:torch.Tensor) -> torch.Tensor:
        return x**3
    
    def forward_symbolic(self, x):
        ## E.g., cvc5.Pow(x,3) would also work here (if cvc5 is the smt solver of choice)
        return x**3
    
    def gradient_symbolic(self, x):
        return 3 * x**2
    
        
class Learner(torch.nn.Module):
    """
    Lyapunov function implemented as torch NN
    params:
    n_dim: dimension of system
    h: width of hidden layer (no impact on function output)
    activation: Activation object
    """
    def __init__(self, n_dim, h, activation):
        super().__init__()
        self.input = torch.nn.Linear(n_dim, h, bias=False)
        self.out = torch.nn.Linear(h, 1, bias=False)
        self.params = (*self.input.parameters(), *self.out.parameters())
        self.activation = activation()
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input(x)
        x = self.activation(x)
        return self.out(x)

    def loss(self, x: torch.Tensor) -> torch.Tensor:
        return NotImplementedError

    def learn(self, S: torch.Tensor, Sdot: torch.Tensor, optimizer: torch.optim.Optimizer) -> None:
        n_error = float('inf') # number of points that do not satisfy the Lyapunov conditions
        while n_error > 0:
            S_clone = torch.clone(S).requires_grad_()
            V = self.forward(S_clone)

            grad_V = torch.autograd.grad(outputs=V, inputs=S_clone,
                                         grad_outputs=torch.ones_like(V),
                                         create_graph=True,
                                         retain_graph=True)[0]
            Vdot = torch.mul(grad_V, Sdot).sum(dim=1)
    
            ###############################
            ########## TODO ############### 
            n_error = None
            loss = None
            ###############################
            ###############################
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

class LyapCegis:
    """
    Basic Cegis implementation:
    params:
    int n: dimension of system
    f: function of dynamics to synthesis Lyapunov function for
    D: Domain of verification
    """
    def __init__(self, n: int, f: Callable, D, activation):
        N_neurons = 5
        self.n_dim = n
        self.f = f
        self.domain = D
        self.learner = Learner(n, N_neurons, activation)
        
        ##########################
        ##### Comment out lines below depening on choice of SMT-solver.
        self.x = [z3.Real('x' + str(i)) for i in range(n)]
        self.x = [cvc5.Real('x' + str(i)) for i in range(n)] 
        ##################
        
    def synthesise(self) -> tuple[bool, Learner]:
        """Function that handles the synthesis of a Lyapunov function given
            the initialisation of the CEGIS object. """
        raise NotImplementedError
            
    def augment_cex(self, S, Sdot, cex):
        """
        Augment data set S, Sdot with cex & f(cex)
        S: torch Tensor
        Sdot: torch Tensor
        cex: model of cex (output of solver.model())
        """
        cex = torch.tensor([float(cex[cex[i]].as_fraction()) for i in range(self.n_dim)]).reshape(1, -1)
        S = torch.cat((S, cex))
        Sdot =  torch.stack([torch.stack(f(*S_x)) for S_x in S])
        return S, Sdot
    
def verify_lyap(V, Vdot, D, x):
    """
    Returns outcome of SMT call & possibly counter-examples
    param:
    V: smt expression for Lyap func
    Vdot: smt expression for Lie derivative of V
    D: domain of verification expressed as radius of hypersphere
    x: smt variables that construct V, Vdot
    
    return: result of verification call (bool),
            counterexamples (either SMT model or None)
    """
    raise NotImplementedError
    
def get_V(learner, x):
    """
    Returns SMT expression of V
    """
    x = np.array(x).reshape(-1, 1)
    act = learner.activation.forward_symbolic
    
    W_in = learner.input.weight.detach().numpy().round(3)
    W_out = learner.out.weight.detach().numpy().round(3)
    return  (W_out @ act(W_in @ x))[0,0]

def get_Vdot(learner, x, f):
    """
    Returns SMT expression of Vdot
    """
    x = np.array(x).reshape(-1, 1)
    act_der = learner.activation.gradient_symbolic
    W_in = learner.input.weight.detach().numpy().round(3)
    W_out = learner.out.weight.detach().numpy().round(3)
    grad_V =  W_out @ np.diagflat(act_der(W_in @  x)) @ W_in
    f_x = np.array(f(*x))
    return (grad_V @ f_x)[0,0]

torch.manual_seed(21)
f = lambda x,y: [-x + x*y, -y]



In [ ]:
# Plotting functionality for V

%matplotlib inline 

from matplotlib import pyplot as plt
from matplotlib import cm

def plot_V(V: Learner):
    fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
    X = np.linspace(-1,1, 20)
    Y = np.linspace(-1,1, 20)
    XX, YY = np.meshgrid(X,Y)
    VV = V(torch.tensor(np.vstack([XX.ravel(),YY.ravel()])).T.float())
    VV = VV.detach().numpy()
    surf = ax.plot_surface(XX,YY, VV.reshape(20,20),  alpha=0.5, cmap=cm.jet)
    plt.show()
    

## Quantifier Alternation

Some SMT solvers are able to reason over formulae with quantifier alternation, and have built in ForALL ($\forall$) functionality. Note, this functionality depends on the theory used and is subject to issues with scaling. 

However, such alternation allows us to directly solve synthesis problems without using CEGIS. We can provide the SMT solver with a parametrised template Lyapunov function and certify stability this way. **Using the candidate $V(x) = ax^2 + by^2$ and the dynamics above, find a Lyapunov function using `z3.ForAll` or `cvc5.ForAll` in the closed unit disc.**

**Does your result make sense?**



### Further Reading: SMT- Solvers for Neural Networks

Recent development in SMT-solving includes SMT-based tools which can perform verification over deep neural networks. These often exploit the piecewise-linearity of ReLU to verify ReLU-based networks, though this is not exclusively the case. Below are some common examples in the literature if you'd like to read further on these.

* [Marabou](https://github.com/NeuralNetworkVerification/Marabou) (built on Reluplex)
* [PLANET](https://github.com/progirep/planet)
* [JAXVerify](https://github.com/deepmind/jax_verify)
* [DNNV](https://github.com/dlshriver/DNNV)


## Further Reading

This session and the CEGIS material draw directly on the following work:

* **Fossil 2.0** -- A. Edwards, A. Peruffo, A. Abate. "Fossil 2.0: Formal Certificate Synthesis for the Verification and Control of Dynamical Models." HSCC '24, ACM, 2024. [doi:10.1145/3641513.3651398](https://doi.org/10.1145/3641513.3651398) | [github.com/oxford-oxcav/fossil](https://github.com/oxford-oxcav/fossil)
* **Neural Abstractions** -- A. Abate, A. Edwards, M. Giacobbe. "Neural Abstractions." NeurIPS 2022.
* **SAT/SMT by Example** -- D. Yurichev. A wide-ranging, example-driven SAT/SMT reference: [https://smt.st/](https://smt.st/) (CC BY-SA 4.0).

Further background:

* D. Kroening, O. Strichman, [*Decision Procedures: An Algorithmic Point of View*](https://www.decision-procedures.org/).
* The [SMT-LIB](https://smtlib.cs.uiowa.edu/) standard.
* L. de Moura, N. Bjørner, *Z3: An Efficient SMT Solver*, TACAS 2008 (the primary Z3 reference).
* P. A. Parrilo, *Semidefinite programming relaxations for semialgebraic problems* (Sum-of-Squares optimisation; the SDP-based alternative to SMT-based certification).

© 2023-2026 Alec Edwards, University of Oxford. Code licensed under the MIT License; written teaching content licensed under CC BY 4.0.

---

This material is dual-licensed: code under the [MIT license](LICENSE) and written teaching content under [CC BY 4.0](LICENSE-CC-BY-4.0). Please cite the repository if you use it [see [`CITATION.cff`](CITATION.cff)].